In [2]:
import json 
import requests

In [3]:
API_KEY = "579b464db66ec23bdd000001d3c9aed4ccb64c8a6e3582ed515b11f4"
BASE_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

In [7]:
def get_mandi_price(state="Maharastra",district=None, commodity=None, limit=10):
    """
    Fetches daily market prices from Govt of India API.
    
    Args:
        state (str): Filter by state (e.g., "Maharashtra")
        district (str): Filter by district (e.g., "Nashik")
        commodity (str): Filter by crop name (e.g., "Onion", "Wheat")
        limit (int): Number of records to return
    """
    params = {
        "api-key": API_KEY,
        "format": "json",
        "limit": limit
    }
    
    # Add filters only if they are provided
    if state:
        params["filters[state.keyword]"] = state
    if district:
        params["filters[district]"] = district
    if commodity:
        params["filters[commodity]"] = commodity

    try:
        print(f"📡 Fetching data for: {commodity or 'All'} in {district or 'All'}...")
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status() # Raise error for 400/403/500 codes
        
        data = response.json()

        print(data)
        # Check if records exist
        records = data.get("records", [])
        if not records:
            return "No data found for the given criteria."
            
        # Format the output nicely for the LLM to read
        formatted_result = []
        for r in records:
            entry = (
                f"State: {r.get('state')}, District: {r.get('district')}, "
                f"Market: {r.get('market')}, Commodity: {r.get('commodity')}, "
                f"Price: ₹{r.get('modal_price')}/Quintal (Date: {r.get('arrival_date')})"
            )
            formatted_result.append(entry)
            
        return "\n".join(formatted_result)

    except requests.exceptions.RequestException as e:
        return f"API Error: {str(e)}"

In [5]:
print("\n--- TEST 1: Onion in Nashik ---")
print(get_mandi_price(state="Maharashtra", district="Nashik", commodity="Onion"))


--- TEST 1: Onion in Nashik ---
📡 Fetching data for: Onion in Nashik...
No data found for the given criteria.


In [8]:
print("\n--- TEST 2: Wheat in Maharashtra ---")
print(get_mandi_price(state="Maharashtra", commodity="Wheat"))


--- TEST 2: Wheat in Maharashtra ---
📡 Fetching data for: Wheat in All...
{'created': 1627939168, 'updated': 1771153267, 'created_date': '2021-08-02T21:19:28Z', 'updated_date': '2026-02-15T11:01:07Z', 'active': '1', 'index_name': '9ef84268-d588-465a-a308-a864a43d0070', 'org': ['Ministry of Agriculture and Farmers Welfare', 'Department of Agriculture and Farmers Welfare'], 'org_type': 'Central', 'source': 'data.gov.in', 'title': 'Current Daily Price of Various Commodities from Various Markets (Mandi)', 'external_ws_url': '', 'visualizable': '1', 'field': [{'name': 'State', 'id': 'state', 'type': 'keyword'}, {'name': 'District', 'id': 'district', 'type': 'keyword'}, {'name': 'Market', 'id': 'market', 'type': 'keyword'}, {'name': 'Commodity', 'id': 'commodity', 'type': 'keyword'}, {'name': 'Variety', 'id': 'variety', 'type': 'keyword'}, {'name': 'Grade', 'id': 'grade', 'type': 'keyword'}, {'name': 'Arrival_Date', 'id': 'arrival_date', 'type': 'date'}, {'name': 'Min_x0020_Price', 'id': 'm

In [9]:
from bs4 import BeautifulSoup 
import urllib3

In [10]:
MSAMB_URL = "https://www.msamb.com/ApmcDetail/APMCPriceInformation#CommodityDistrictGird"

In [26]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time


In [40]:
import selenium
selenium.__version__

'4.40.0'

In [46]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

# Specify the path to your ChromeDriver executable
 # Example path for macOS/Linux

brave_path = '/opt/brave.com/brave/brave'

# Set up Chrome options and specify the binary location
options = webdriver.ChromeOptions()
options.binary_location = brave_path

# Initialize the WebDriver with the specified options and service
service = Service()
driver = webdriver.Chrome(options = options)
# Now you can run your tests as usual
driver.get(MSAMB_URL)
print(driver.title)



Titles


In [47]:
wait = WebDriverWait(driver,10)

In [48]:
lang_element = wait.until(EC.element_to_be_clickable((By.ID,"language")))
lang_select = Select(lang_element)

In [49]:
if "English" not in lang_select.first_selected_option.text:
        print("🌐 Switching to English...")
        lang_select.select_by_visible_text("English")
        time.sleep(3) # Wait for reload
        wait.until(EC.presence_of_element_located((By.ID, "drpCommodities")))

🌐 Switching to English...


In [51]:
print("selecting commodity")

dropdown = Select(driver.find_element(By.ID,"drpCommodities"))

selecting commodity


In [63]:
found=False
crop_name ="wheat"
for opt in dropdown.options:
    if crop_name.upper() in opt.text.upper():
        dropdown.select_by_visible_text(opt.text)
        found=True
        break

    

In [67]:
table_element = driver.find_element(By.CSS_SELECTOR, "table.table.custom-table")
table_html = table_element.get_attribute('outerHTML')

In [68]:
import pandas as pd 
from io import StringIO
df = pd.read_html(StringIO(table_html))[0]

In [69]:
df

,APMC,Variety,Unit,Quantity,Lrate,Hrate,Modal
0,15/02/2026,15/02/2026,15/02/2026,15/02/2026,15/02/2026,15/02/2026,15/02/2026
1,RAHURI,----,QUINTAL,22,2450,2500,2475
2,PAITHAN,BANSI,QUINTAL,5,2600,2600,2600
3,BULDHANA,HYBRID,QUINTAL,9,1800,2200,2000
4,14/02/2026,14/02/2026,14/02/2026,14/02/2026,14/02/2026,14/02/2026,14/02/2026
...,...,...,...,...,...,...,...
443,KANDHAR,LOCAL,QUINTAL,21,2500,2800,2600
444,BALAPUR,LOCAL,QUINTAL,15,2000,3100,2450
445,KATOL,LOCAL,QUINTAL,2,2400,2400,2400
446,SHIRUR,No. 2,QUINTAL,15,2200,2250,2250


In [34]:


# 1. Run in background (Fixes "DevToolsActivePort" & GUI errors)
# chrome_options.add_argument("--headless=new") 
# # 2. Required for most server/notebook environments
# chrome_options.add_argument("--no-sandbox")
# # 3. Fixes memory issues on some systems
# chrome_options.add_argument("--disable-dev-shm-usage")

# --- INITIALIZE DRIVER ---
# This automatically downloads the correct driver for your installed Chrome version
brave_path = '/opt/brave.com/brave/brave'

# Set up Chrome options and specify the binary locatio

try:
    url =MSAMB_URL
    print("Opening url")
    driver.get(url)

    wait = WebDriverWait(driver, 15)

    lang_element = wait.until(EC.element_to_be_clickable((By.ID,"language")))
    lang_select = Select(lang_element)

    if "English" not in lang_select.first_selected_option.text:
        print("🌐 Switching to English...")
        lang_select.select_by_visible_text("English")
        time.sleep(3) # Wait for reload
        wait.until(EC.presence_of_element_located((By.ID, "drpCommodities")))

    # 3. Select Commodity
    print(f"🔍 Selecting '{crop_name}'...")
    dropdown = Select(driver.find_element(By.ID, "drpCommodities"))
    found = False
    for opt in dropdown.options:
        if crop_name.upper() in opt.text.upper():
            dropdown.select_by_visible_text(opt.text)
            found = True
            break

    time.sleep(3) # Let the table populate
        
    # Locate all rows in the specific commodity table body
    rows = driver.find_elements(By.XPATH, "//tbody[@id='tblCommodity']/tr")
    print(f"📥 Found {len(rows)} market entries. Extracting...")
    
    full_data = []
    
    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        # Ensure it's a valid data row (sometimes headers repeat or empty rows exist)
        if len(cols) >= 7:
            entry = {
                "APMC": cols[0].text.strip(),
                "Variety": cols[1].text.strip(),
                "Unit": cols[2].text.strip(),
                "Quantity": cols[3].text.strip(),
                "Min_Price": cols[4].text.strip(),
                "Max_Price": cols[5].text.strip(),
                "Modal_Price": cols[6].text.strip(),
                "Date": "Today" # You can scrape the date header if needed
            }
            full_data.append(entry)

except Exception as e:
    print(e)

WebDriverException: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x5bc24fe674e3 <unknown>
#1 0x5bc24fb96c76 <unknown>
#2 0x5bc24fbbd757 <unknown>
#3 0x5bc24fbbc029 <unknown>
#4 0x5bc24fbfaccc <unknown>
#5 0x5bc24fbfa47f <unknown>
#6 0x5bc24fbf1de3 <unknown>
#7 0x5bc24fbc72dd <unknown>
#8 0x5bc24fbc834e <unknown>
#9 0x5bc24fe273e4 <unknown>
#10 0x5bc24fe2b3d7 <unknown>
#11 0x5bc24fe35b20 <unknown>
#12 0x5bc24fe2c023 <unknown>
#13 0x5bc24fdfa1aa <unknown>
#14 0x5bc24fe506b8 <unknown>
#15 0x5bc24fe50847 <unknown>
#16 0x5bc24fe60243 <unknown>
#17 0x7c4eae89caa4 <unknown>
#18 0x7c4eae929c6c <unknown>
